# TrustSkin AI — Turnkey Google Colab GPU Training & Evaluation Notebook

**CS320 Fall 2026 — Project P08: Reliable Recognition of Difficult Skin Lesions**  
**Repository:** https://github.com/SikandarHussain6858/TrustSkinAI

### Before Running
1. In Colab, go to **Runtime → Change runtime type → T4 GPU** (or A100/L4 if available).
2. Run all cells top-to-bottom. The notebook will:
   - Mount Google Drive (only to back up trained `checkpoints/` and `outputs/`).
   - Clone `TrustSkinAI` and install `trustskin` in editable mode.
   - **Download & extract MedIMeta `derm` directly from Zenodo** onto Colab's fast local SSD (no manual 1.8 GB upload needed!).
   - Run all 4 controlled experiments (`BaselineCNN`, `BaselineCNN + Temp Scaling`, `UncertaintyCNN + MC Dropout + Temp Scaling`, and comparative `ResNet18MC + MC Dropout + Temp Scaling`).
   - Generate all comparison tables, Risk–Coverage curves, training curves, and difficult-case visual grids.
   - Automatically back up `checkpoints/` and `outputs/` to Google Drive.

In [ ]:
# Cell 1: Verify GPU runtime
!nvidia-smi

In [ ]:
# Cell 2: Mount Google Drive (for saving trained checkpoints & outputs)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS_DIR = '/content/drive/MyDrive/TrustSkinAI/results'
!mkdir -p {DRIVE_RESULTS_DIR}
print('Drive backup directory ready:', DRIVE_RESULTS_DIR)

In [ ]:
# Cell 3: Clone TrustSkinAI repo (or pull latest if already cloned) and install package
import os

REPO_URL = 'https://github.com/SikandarHussain6858/TrustSkinAI.git'
REPO_DIR = '/content/TrustSkinAI'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd /content/TrustSkinAI
!pip install -q medimeta PyYAML tqdm
!pip install -q -e .

In [ ]:
# Cell 4: Direct download & extraction of MedIMeta Dermatoscopy (`derm`) from Zenodo onto Colab SSD
import json
import urllib.request
import zipfile
from pathlib import Path

derm_dir = Path('/content/TrustSkinAI/data/derm')
zip_path = Path('/content/derm.zip')

if (derm_dir / 'info.yaml').exists():
    print('MedIMeta derm dataset already extracted at:', derm_dir)
else:
    print('Querying Zenodo record 7884735 for MedIMeta derm archive...')
    with urllib.request.urlopen('https://zenodo.org/api/records/7884735') as resp:
        record = json.loads(resp.read().decode('utf-8'))
    files = record.get('files', [])
    zip_file = next((f for f in files if f['key'].endswith('.zip')), files[0])
    download_url = zip_file['links']['self']
    print(f"Downloading {zip_file['key']} ({zip_file['size'] / 1e9:.2f} GB) from {download_url} ...")
    !wget -q --show-progress -O /content/derm.zip "{download_url}"

    print('Extracting into /content/TrustSkinAI/data/ ...')
    extract_root = Path('/content/TrustSkinAI/data')
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        top_names = {name.split('/')[0] for name in zf.namelist() if name}
        target = extract_root if 'derm' in top_names else derm_dir
        target.mkdir(parents=True, exist_ok=True)
        zf.extractall(target)
    zip_path.unlink(missing_ok=True)
    print('Extraction complete!')

!ls -lh /content/TrustSkinAI/data/derm

In [ ]:
# Cell 5: Verify all unit tests & real-dataset smoke test pass on Colab
!pytest tests/ -v

In [ ]:
# Cell 6: Exp 0 — Train & Evaluate CS320 Reference Baseline CNN (3-block scratch, CE, no aug, 20 epochs)
!python scripts/02_train_baseline.py --config configs/baseline_cnn.yaml

In [ ]:
# Cell 7: Exp 0-Cal — Controlled Ablation: Does Temperature Scaling alone fix BaselineCNN?
!python scripts/04_calibrate_and_select.py \
    --config configs/baseline_cnn.yaml \
    --checkpoint checkpoints/baseline_cnn_best.pt \
    --mc-samples 1 \
    --exp-name baseline_cnn_calibrated

In [ ]:
# Cell 8: Exp 1/2/3 — Train Main Proposed TrustSkin AI Model (UncertaintyCNN + Aug + Weighted CE)
!python scripts/03_train_improved.py --config configs/improved_mc_cnn.yaml

# Fit Temperature Scaling on Val + Evaluate MC Dropout (T=25) & Selective Risk on Test
!python scripts/04_calibrate_and_select.py \
    --config configs/improved_mc_cnn.yaml \
    --checkpoint checkpoints/improved_mc_cnn_best.pt

In [ ]:
# Cell 9: Exp 4 — Comparative Reference: Pretrained ResNet-18 + MC Dropout (T=25) + Temperature Scaling
!python scripts/03_train_improved.py --config configs/resnet18_mc.yaml

!python scripts/04_calibrate_and_select.py \
    --config configs/resnet18_mc.yaml \
    --checkpoint checkpoints/resnet18_mc_best.pt

In [ ]:
# Cell 10: Exp 5 & 6 — Compile Comparison Table, Combined Risk-Coverage Plot, Training Curves & Difficult-Case Grids
!python scripts/05_compare_experiments.py --output-dir outputs

!python scripts/06_error_analysis.py --config configs/base.yaml --predictions outputs/tables/baseline_cnn_test_predictions.csv
!python scripts/06_error_analysis.py --config configs/base.yaml --predictions outputs/tables/improved_mc_cnn_test_predictions.csv
!python scripts/06_error_analysis.py --config configs/base.yaml --predictions outputs/tables/resnet18_mc_test_predictions.csv

In [ ]:
# Cell 11: Back up all trained checkpoints & outputs to Google Drive (+ create downloadable zip)
!cp -r checkpoints outputs /content/drive/MyDrive/TrustSkinAI/results/
!zip -rq /content/trustskin_colab_results.zip checkpoints outputs
!cp /content/trustskin_colab_results.zip /content/drive/MyDrive/TrustSkinAI/results/
print('All checkpoints and outputs backed up to /content/drive/MyDrive/TrustSkinAI/results/')

# Optional: Display the final controlled comparison table right here
import pandas as pd
display(pd.read_csv('outputs/tables/experiment_comparison.csv'))